## Setting up

In [1]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install git+https://github.com/huggingface/transformers.git

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "garbage_collection_threshold:0.6,max_split_size_mb:128"

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HUGGINGFACE_TOKEN")

import subprocess
subprocess.run(["pip", "install", "-q", "--upgrade", "huggingface_hub"], check=True)

from huggingface_hub import login
login(hf_token)

In [ ]:
import wandb

wb_token = user_secrets.get_secret("wandb")

wandb.login(key=wb_token)
run = wandb.init(
    project='covalent-gemma-lora', 
    job_type="training", 
    anonymous="allow"
)

## Loading the model and tokenizer

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
    
)

## Model inference before fine-tuning

In [ ]:
prompt_style = """{}"""

In [ ]:
sample_prompt = """You are a startup ecosystem matching engine. Score each candidate for compatibility with the viewer.
VIEWER:
{"id": "founder-aisha", "type": "founder", "name": "Aisha Razak", "company": "PayChain", "sector": ["fintech", "payments"], "stage": "seed", "geography": "Kuala Lumpur", "remote_friendly": true}

CANDIDATES:
[{"id": "mentor-james", "type": "mentor", "name": "James Foo", "expertise": ["fintech", "payments"], "stage_preference": ["seed"], "geography": "Kuala Lumpur", "remote_friendly": true, "availability": "high"}]

Score each candidate 0-100 across domain_fit, stage_fit, geography, history, availability dimensions.
Return ONLY a valid JSON array:
[{"id":"...","score":87,"rationale":"...","flags":["✗ ..."],"breakdown":{"domain_fit":26,"stage_fit":22,"geography":12,"history":18,"availability":9}}]"""

FastModel.for_inference(model)
inputs = tokenizer([prompt_style.format(sample_prompt)], return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask, max_new_tokens=512, use_cache=True)
response = tokenizer.batch_decode(outputs)
print(response[0])

In [8]:
torch.cuda.empty_cache() 

In [9]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3417,
)

## Loading and processing the dataset

In [ ]:
train_prompt_style = """{}{}"""

In [ ]:
import json

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    prompts = examples["prompt"]
    completions = examples["completion"]
    texts = []
    for prompt, completion in zip(prompts, completions):
        text = train_prompt_style.format(prompt, completion) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

from datasets import load_dataset
dataset = load_dataset("json", data_files="/kaggle/input/datasets/yihui06/myhack2026/training-data.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Loaded {len(dataset)} training examples")
print(dataset["text"][0][:500])

## Setting up the model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=200,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3417,
        output_dir="outputs",
    ),
)

## Model training

In [14]:
torch.cuda.empty_cache()
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 14,901,248/4,000,000,000 (0.37% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
It is strongly recommended to train Gemma3 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,2.554000
20,2.586200
30,2.491500
40,2.515900
50,2.549800
60,2.579800


In [15]:
# Save the fine-tuned model
wandb.finish()

wandb:                                                                                
wandb: 
wandb: Run history:
wandb:         train/epoch ▁▂▄▅▇██
wandb:   train/global_step ▁▂▄▅▇██
wandb: train/learning_rate █▇▅▄▂▁
wandb:          train/loss ▆█▁▃▅█
wandb: 
wandb: Run summary:
wandb:               total_flos 5365938066554880.0
wandb:              train/epoch 0.48
wandb:        train/global_step 60
wandb:          train/grad_norm nan
wandb:      train/learning_rate 0
wandb:               train/loss 2.5798
wandb:               train_loss 2.54621
wandb:            train_runtime 1843.6797
wandb: train_samples_per_second 0.13
wandb:   train_steps_per_second 0.033
wandb: 
wandb: 🚀 View run mild-glitter-1 at: https://wandb.ai/kingabzpro/Fine-tuning%20Gemma-3-4B%20on%20FinQA%20Reasoning%20Dataset/runs/8gi47vqw
wandb: ⭐️ View project at: https://wandb.ai/kingabzpro/Fine-tuning%20Gemma-3-4B%20on%20FinQA%20Reasoning%20Dataset
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and

## Model inference after fine-tuning

In [ ]:
test_prompt = """You are a startup ecosystem matching engine. Score each candidate for compatibility with the viewer.
VIEWER:
{"id": "founder-daniel", "type": "founder", "name": "Daniel Lim", "company": "MedSync", "sector": ["healthtech", "ai"], "stage": "pre-seed", "geography": "Penang", "remote_friendly": true}

CANDIDATES:
[{"id": "mentor-sarah", "type": "mentor", "name": "Sarah Tan", "expertise": ["healthtech", "product-strategy"], "stage_preference": ["pre-seed"], "geography": "Penang", "remote_friendly": true, "availability": "medium"},
{"id": "mentor-james", "type": "mentor", "name": "James Foo", "expertise": ["fintech", "payments"], "stage_preference": ["seed"], "geography": "Kuala Lumpur", "remote_friendly": true, "availability": "high"}]

Score each candidate 0-100. Return ONLY a valid JSON array:
[{"id":"...","score":87,"rationale":"...","flags":["✗ ..."],"breakdown":{"domain_fit":26,"stage_fit":22,"geography":12,"history":18,"availability":9}}]"""

FastModel.for_inference(model)
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask, max_new_tokens=512, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

## Saving the model locally

In [ ]:
new_model_online = "YOUR_HF_USERNAME/covalent-gemma-lora"
new_model_local = "covalent-gemma-lora"
model.save_pretrained(new_model_local)
tokenizer.save_pretrained(new_model_local)
print(f"Adapter saved to ./{new_model_local}/")
print("Download this folder and copy contents to deploy/ollama/adapters/covalent-lora/")

## Pushing the model to Hugging Face hub

In [19]:
model.push_to_hub(new_model_online) # Online saving
tokenizer.push_to_hub(new_model_online) # Online saving

README.md:   0%|          | 0.00/601 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/59.7M [00:00<?, ?B/s]

Saved model to https://huggingface.co/kingabzpro/Gemma-3-4B-Fin-QA-Reasoning


tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

In [20]:
model.push_to_hub_merged(new_model_online, tokenizer, save_method = "merged_16bit")

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [02:54<02:54, 174.11s/it]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [04:55<00:00, 147.78s/it]
